# **ColabProSST**

ColabProSST brings the ProSST backbone into the SaprotHub workflow with a compact SaProt-style entry point. ProSST is not SaProt: it uses amino-acid `input_ids` together with official ProSST `ss_input_ids`, so every task needs either `structure_tokens`, `pdb_path`/`structure_path`, or structure tokens generated in this notebook.

Supported in this first version: structure-token conversion, zero-shot mutation effect prediction, protein-level classification fine-tuning, protein-level regression fine-tuning, checkpoint prediction, and optional Hugging Face upload.


In [ ]:
#@title **Click the run-button to prepare ColabProSST**
import os
import subprocess
import sys
from pathlib import Path

SAPROTHUB_REPO = 'https://github.com/Hello-github-code/SaprotHub' #@param {type:'string'}
SAPROTHUB_BRANCH = 'prosst' #@param {type:'string'}
DOWNLOAD_CSV_TEMPLATES = False #@param {type:'boolean'}

ROOT = Path('/content')
SAPROT_HOME = ROOT / 'SaprotHub'
PROSST_HOME = ROOT / 'ProSST'
PYG_HOME = ROOT / '.cache' / 'pyg'
HF_HOME = ROOT / '.cache' / 'huggingface'
INSTALL_MARKER = ROOT / '.colabprosst_deps_installed'
INSTALL_MARKER_VERSION = 'preserve-colab-torch-v1'
for path in [PYG_HOME, HF_HOME, ROOT / 'prosst_uploads', ROOT / 'prosst_structure_assets']:
    path.mkdir(parents=True, exist_ok=True)
os.environ['PYG_HOME'] = str(PYG_HOME)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

if not SAPROT_HOME.exists():
    try:
        subprocess.run(['git', 'clone', '--depth', '1', '-b', SAPROTHUB_BRANCH, SAPROTHUB_REPO, str(SAPROT_HOME)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f'Could not clone {SAPROTHUB_REPO}@{SAPROTHUB_BRANCH}. '
            'Push the ColabProSST branch or set SAPROTHUB_REPO/SAPROTHUB_BRANCH '
            'to a repository that contains this notebook and the ProSST modules.'
        ) from exc
else:
    subprocess.run(['git', '-C', str(SAPROT_HOME), 'fetch', 'origin', SAPROTHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(SAPROT_HOME), 'checkout', SAPROTHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(SAPROT_HOME), 'reset', '--hard', f'origin/{SAPROTHUB_BRANCH}'], check=True)
if not PROSST_HOME.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ai4protein/ProSST', str(PROSST_HOME)], check=True)

def run_pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', *args], check=True)

def probe_torch():
    code = "import torch; print(torch.__version__); print(torch.version.cuda or 'cpu'); print(float(torch.zeros(1)[0]))"
    return subprocess.run([sys.executable, '-c', code], text=True, capture_output=True)

def write_requirements_without_colab_torch():
    source = SAPROT_HOME / 'requirements.txt'
    target = ROOT / 'colabprosst_requirements_without_torch.txt'
    skipped = ('torch==', 'torchvision==', 'torchaudio==')
    lines = []
    for raw_line in source.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if line.startswith(skipped):
            continue
        lines.append(raw_line)
    target.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return target

def install_pyg_extensions(torch_version, pyg_backend):
    version_candidates = [torch_version]
    version_parts = torch_version.split('.')
    if len(version_parts) >= 2:
        version_candidates.append(f'{version_parts[0]}.{version_parts[1]}.0')
    tried_urls = []
    last_error = None
    for version in dict.fromkeys(version_candidates):
        wheel_url = f'https://data.pyg.org/whl/torch-{version}+{pyg_backend}.html'
        tried_urls.append(wheel_url)
        try:
            run_pip('install', '--no-index', 'torch-scatter', 'torch-sparse', 'torch-cluster', 'torch-spline-conv', '-f', wheel_url)
            return wheel_url
        except subprocess.CalledProcessError as exc:
            last_error = exc
    raise RuntimeError(f'Could not install PyG extension wheels from {tried_urls}') from last_error

marker_value = INSTALL_MARKER.read_text(encoding='utf-8').strip() if INSTALL_MARKER.exists() else ''
if marker_value != INSTALL_MARKER_VERSION:
    torch_probe = probe_torch()
    if torch_probe.returncode != 0:
        raise RuntimeError(
            'The current Colab runtime already has a broken PyTorch install, usually from an earlier run that overwrote Colab torch.\n'
            'Please use Runtime -> Disconnect and delete runtime, reopen this updated notebook, and run the first cell again.\n\n'
            f'STDOUT:\n{torch_probe.stdout}\nSTDERR:\n{torch_probe.stderr}'
        )
    torch_lines = torch_probe.stdout.strip().splitlines()
    torch_version = torch_lines[0].split('+')[0]
    torch_cuda = torch_lines[1]

    filtered_requirements = write_requirements_without_colab_torch()
    run_pip('install', '-r', str(filtered_requirements))
    # Colab often preloads NumPy. Reinstall binary packages, then restart so pandas loads the same ABI.
    run_pip('install', '--force-reinstall', '--no-cache-dir', 'numpy==1.26.4', 'pandas==2.2.2', 'scipy==1.15.1', 'scikit-learn==1.4.0')
    run_pip('install', 'joblib==1.3.2', 'pathos==0.3.2', 'biotite==0.39.0')
    pyg_backend = 'cpu' if torch_cuda == 'cpu' else 'cu' + torch_cuda.replace('.', '')
    pyg_wheel_url = install_pyg_extensions(torch_version, pyg_backend)
    run_pip('install', 'torch-geometric==2.5.0')
    INSTALL_MARKER.write_text(INSTALL_MARKER_VERSION, encoding='utf-8')
    print('ColabProSST dependencies installed without replacing Colab torch. Restarting runtime once so binary wheels load cleanly.')
    try:
        from google.colab import runtime
        runtime.restart_runtime()
    except Exception:
        os.kill(os.getpid(), 9)

os.environ['PROSST_HOME'] = str(PROSST_HOME)
for p in [str(SAPROT_HOME), str(SAPROT_HOME / 'saprot'), str(PROSST_HOME)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import torch
print('NumPy/Pandas ABI check:', np.__version__, pd.__version__)
print('Torch check:', torch.__version__, torch.version.cuda or 'cpu')

from saprot.utils.colab_prosst_workflow import ColabProSSTWorkflow

COLABPROSST_WORKFLOW = ColabProSSTWorkflow()
COLABPROSST_WORKFLOW.create_csv_templates(download=DOWNLOAD_CSV_TEMPLATES)
print('ColabProSST is ready.')


In [ ]:
#@title **Run ColabProSST**
from IPython.display import display

WORKFLOW = 'Convert structure to tokens' #@param ['Convert structure to tokens', 'Zero-shot mutation effect prediction', 'Fine-tune classification', 'Fine-tune regression', 'Predict classification', 'Predict regression']
INPUT_CSV = '' #@param {type:'string'}
UPLOAD_INPUT_CSV = False #@param {type:'boolean'}
STRUCTURE_FILE = '' #@param {type:'string'}
UPLOAD_STRUCTURE_FILE = False #@param {type:'boolean'}
CHAIN_ID = '' #@param {type:'string'}
USE_LAST_STRUCTURE_TOKENS = False #@param {type:'boolean'}
STRUCTURE_ZIP = '' #@param {type:'string'}
UPLOAD_STRUCTURE_ZIP = False #@param {type:'boolean'}
CHECKPOINT_PATH = '' #@param {type:'string'}
TASK_NAME = 'ProSSTUserTask' #@param {type:'string'}
NUM_LABELS = 2 #@param {type:'integer'}
MAX_EPOCHS = 2 #@param {type:'integer'}
BATCH_SIZE = 1 #@param {type:'integer'}
FREEZE_BACKBONE = False #@param {type:'boolean'}
DOWNLOAD_OUTPUTS = True #@param {type:'boolean'}

# Advanced defaults. Most users can leave these unchanged for the first ProSST-2048 version.
MODEL_PATH = 'AI4Protein/ProSST-2048' #@param {type:'string'}
STRUCTURE_VOCAB_SIZE = 2048 #@param {type:'integer'}
GRADIENT_CHECKPOINTING = True #@param {type:'boolean'}
OUTPUT_DIR = '/content/colabprosst_outputs' #@param {type:'string'}

if 'COLABPROSST_WORKFLOW' not in globals():
    raise RuntimeError('Run the preparation cell first.')

COLABPROSST_WORKFLOW.set_output_dir(OUTPUT_DIR)

if WORKFLOW == 'Convert structure to tokens':
    result_df = COLABPROSST_WORKFLOW.convert_structure(
        structure_path=STRUCTURE_FILE,
        upload_structure=UPLOAD_STRUCTURE_FILE,
        chain_id=CHAIN_ID,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        download=DOWNLOAD_OUTPUTS,
    )
    display(result_df)

elif WORKFLOW == 'Zero-shot mutation effect prediction':
    result_df = COLABPROSST_WORKFLOW.run_zero_shot(
        input_csv=INPUT_CSV,
        upload_csv=UPLOAD_INPUT_CSV,
        use_last_structure_tokens=USE_LAST_STRUCTURE_TOKENS,
        structure_zip=STRUCTURE_ZIP,
        upload_structure_zip=UPLOAD_STRUCTURE_ZIP,
        model_path=MODEL_PATH,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        download=DOWNLOAD_OUTPUTS,
    )
    display(result_df.head())

elif WORKFLOW in ['Fine-tune classification', 'Fine-tune regression']:
    task_type = 'classification' if WORKFLOW.endswith('classification') else 'regression'
    result = COLABPROSST_WORKFLOW.train_downstream(
        task_type=task_type,
        input_csv=INPUT_CSV,
        upload_csv=UPLOAD_INPUT_CSV,
        use_last_structure_tokens=USE_LAST_STRUCTURE_TOKENS,
        structure_zip=STRUCTURE_ZIP,
        upload_structure_zip=UPLOAD_STRUCTURE_ZIP,
        task_name=TASK_NAME,
        num_labels=NUM_LABELS,
        max_epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        model_path=MODEL_PATH,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        freeze_backbone=FREEZE_BACKBONE,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        download=DOWNLOAD_OUTPUTS,
    )
    print(result)

elif WORKFLOW in ['Predict classification', 'Predict regression']:
    task_type = 'classification' if WORKFLOW.endswith('classification') else 'regression'
    if not CHECKPOINT_PATH.strip():
        raise ValueError('Set CHECKPOINT_PATH before running prediction.')
    result_df = COLABPROSST_WORKFLOW.predict_downstream(
        task_type=task_type,
        input_csv=INPUT_CSV,
        checkpoint_path=CHECKPOINT_PATH,
        upload_csv=UPLOAD_INPUT_CSV,
        use_last_structure_tokens=USE_LAST_STRUCTURE_TOKENS,
        structure_zip=STRUCTURE_ZIP,
        upload_structure_zip=UPLOAD_STRUCTURE_ZIP,
        num_labels=NUM_LABELS,
        batch_size=BATCH_SIZE,
        model_path=MODEL_PATH,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        download=DOWNLOAD_OUTPUTS,
    )
    display(result_df.head())


In [ ]:
#@title **Optional: upload a trained ColabProSST checkpoint to Hugging Face**
HF_REPO_ID = '' #@param {type:'string'}
HF_PRIVATE_REPO = False #@param {type:'boolean'}
RUN_HF_LOGIN = True #@param {type:'boolean'}
CHECKPOINT_PATH = '/content/SaprotHub/weights/prosst/ProSSTUserTask.pt' #@param {type:'string'}
TASK_TYPE = 'classification' #@param ['classification', 'regression']
NUM_LABELS = 2 #@param {type:'integer'}
MODEL_PATH = 'AI4Protein/ProSST-2048' #@param {type:'string'}
MODEL_CARD_TITLE = 'ColabProSST model' #@param {type:'string'}
MODEL_DESCRIPTION = 'A ProSST checkpoint trained with ColabProSST.' #@param {type:'string'}
DOWNLOAD_MODEL_PACKAGE = False #@param {type:'boolean'}

if 'COLABPROSST_WORKFLOW' not in globals():
    raise RuntimeError('Run the preparation cell first.')

package_dir = COLABPROSST_WORKFLOW.upload_checkpoint_to_hf(
    repo_id=HF_REPO_ID,
    checkpoint_path=CHECKPOINT_PATH,
    task_type=TASK_TYPE,
    num_labels=NUM_LABELS,
    model_path=MODEL_PATH,
    private=HF_PRIVATE_REPO,
    run_login=RUN_HF_LOGIN,
    title=MODEL_CARD_TITLE,
    description=MODEL_DESCRIPTION,
    download_package=DOWNLOAD_MODEL_PACKAGE,
)
print('local package:', package_dir)
